In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.geo import add_geo_features
from pathlib import Path

sns.set_style("whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
df = pd.read_csv("data/raw/nekretnine_dataset.csv")

df.head()

In [ ]:
print(df.shape)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# square footage cleanup
df["Square_footage"] = (
    df["Square_footage"]
    .str.replace(" m²", "", regex=False)
    .astype(float)
)

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
#remove missing price 
df = df.dropna(subset=["Price_EUR"])
#remove missing sqaure_footage
df = df.dropna(subset=["Square_footage"])

In [ ]:
# fill missing state values
df["State"] = df["State"].fillna("Standardna gradnja")

In [ ]:
#fill missing heating values
df["Heating"] = df["Heating"].fillna("Centralno grejanje")

In [ ]:
median_m2_per_room = (df["Square_footage"] / df["Number_of_rooms"]).median()
print(median_m2_per_room)

#fill missing room numbers with rounded value of square footage and median of m^2 per room
df.loc[df["Number_of_rooms"].isnull(), "Number_of_rooms"] = (
    df["Square_footage"] / median_m2_per_room
).round()

In [ ]:
#Checking outliers 
plt.figure(figsize=(8,4))
sns.boxplot(x=df["Price_EUR"])
plt.title("Price Outliers")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x=df["Square_footage"])
plt.title("Square Footage Outliers")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x=df["Number_of_rooms"])
plt.title("Room Count Outliers")
plt.show()

In [ ]:
df.sort_values("Price_EUR", ascending=False).head(10)

In [ ]:
df.sort_values("Square_footage", ascending=False).head(10)

In [ ]:
df.sort_values("Number_of_rooms", ascending=False).head(10)

In [ ]:
# droping extreme outliers
df = df[(df["Square_footage"] >= 15) & (df["Square_footage"] <= 400)]
df = df[(df["Number_of_rooms"] > 0) & (df["Number_of_rooms"] <= 10)]

In [ ]:
min_price = 20000
max_price = 2000000
df = df[(df["Price_EUR"] >= min_price) & (df["Price_EUR"] <= max_price)]

In [ ]:
# Create price per m2 feature
df["price_per_m2"] = df["Price_EUR"] / df["Square_footage"]

# Drop apartments where price_per_m2 > 11,000
df = df[df["price_per_m2"] <= 11000]

# Check the new dataset
print(df.describe())

In [ ]:
#dropping outliers for small apartments with large amount of rooms
df.loc[df["Square_footage"] < 70, "Number_of_rooms"] = df.loc[df["Square_footage"] < 70, "Number_of_rooms"].clip(upper=4)

In [ ]:
plt.figure(figsize=(14,6))
sns.histplot(df["Price_EUR"], bins=50, kde=True)
plt.title("Distribution of Price")
plt.show()

sns.histplot(df["price_per_m2"], bins=50, kde=True)
plt.title("Distribution of price_per_m2")
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

corr = df.corr(numeric_only=True)

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Feature Correlation Heatmap", fontsize=16)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    x="Square_footage",
    y="Price_EUR",
    data=df,
    alpha=0.5
)

plt.title("Price vs Square Footage", fontsize=16)
plt.xlabel("Square Footage (m²)")
plt.ylabel("Price (EUR)")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(
    df["price_per_m2"],
    bins=40,
    kde=True
)

plt.title("Distribution of Price per m²", fontsize=16)
plt.xlabel("Price per m² (EUR)")
plt.show()

In [ ]:
df["State"].unique()

In [ ]:
df["Heating"].unique()

In [ ]:
df["Street"].unique()

In [ ]:
#Categorize the state feature
state_map = {
    "Novogradnja": "New",
    "U izgradnji": "New",
    "Završena izgradnja": "New",
    "U pripremi": "New",

    "Standardna gradnja": "Standard",

    "Izvorno stanje": "Needs_renovation",

    "Kompletna rekonstrukcija": "Renovated",
    "Delimična rekonstrukcija": "Renovated",

    "Lux": "Luxury"
}

df["State"] = df["State"].map(state_map)

In [ ]:
# Heating categorization
heating_types = [
    "Centralno grejanje",
    "Etažno grejanje na gas",
    "Etažno grejanje na struju",
    "Etažno grejanje na čvrsto gorivo",
    "Podno grejanje",
    "Toplotna pumpa",
    "TA peć",
    "Kamin",
    "Peć na drva/ugalj",
    "Klima uređaj"
]

for heating in heating_types:
    df[f"heating_{heating}"] = df["Heating"].str.contains(heating, na=False).astype(int)

df.drop(columns=["Heating"], inplace=True)

In [ ]:
df = add_geo_features(
    df,
    location_col="Street",
    cache_path=Path("geo_cache.csv"),
    user_agent="belgrade_real_estate_model",
    city_hint="Belgrade",
    country_hint="Serbia",
    delay_seconds=1,
    center_lat=44.8167,
    center_lon=20.4600
)

In [ ]:
# fix broken coordinates
df = df[
    (df["lat"].between(44.6, 45.0)) &
    (df["lon"].between(20.2, 20.7))
]

In [ ]:
# drop this column since its not neccessary(0 missing after droping)
df = df.drop(columns=["geo_missing"])


In [ ]:
sns.scatterplot(x="dist_to_center_km", y="Price_EUR", data=df)

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df, x="State", y="price_per_m2")
plt.xticks(rotation=45)
plt.title("Price per m² by Apartment State")
plt.show()

In [ ]:

plt.figure(figsize=(10,8))

plt.scatter(
    df["lon"],
    df["lat"],
    c=df["price_per_m2"],
    cmap="viridis",
    alpha=0.7
)

plt.colorbar(label="Price per m² (EUR)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Apartment Price per m² across Belgrade")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x="dist_to_center_km",
    y="price_per_m2",
    alpha=0.6
)

sns.regplot(
    data=df,
    x="dist_to_center_km",
    y="price_per_m2",
    scatter=False,
    color="red"
)

plt.title("Price per m² vs Distance to Belgrade Center")
plt.xlabel("Distance to Center (km)")
plt.ylabel("Price per m² (EUR)")

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

plt.hexbin(
    df["lon"],
    df["lat"],
    C=df["price_per_m2"],
    gridsize=40,
    cmap="plasma"
)

plt.colorbar(label="Average price per m²")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Geographic Heatmap of Apartment Prices")

plt.show()

In [ ]:
print(df.columns)

In [ ]:
df["zone"] = pd.cut(
    df["dist_to_center_km"],
    bins=[0,3,6,10,25],
    labels=["Center","Inner city","Outer city","Suburbs"]
)

heating_cols = [c for c in df.columns if c.startswith("heating_")]

heating_zone = df.groupby("zone")[heating_cols].sum()

heating_zone.plot(
    kind="bar",
    stacked=True,
    figsize=(12,7),
    colormap="viridis"
)

plt.title("Heating Types by Distance Zone")
plt.ylabel("Number of Apartments")

plt.show()

In [ ]:
plt.figure(figsize=(10,8))

plt.scatter(
    df["lon"],
    df["lat"],
    c=df["Optical_internet"],
    cmap="coolwarm",
    alpha=0.6
)

plt.colorbar(label="Optical Internet")
plt.title("Geographic Distribution of Optical Internet")

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [ ]:
print(df[["dist_to_center_km","Optical_internet"]].corr())

In [ ]:
df["zone"] = pd.cut(
    df["dist_to_center_km"],
    bins=[0,3,6,10,25],
    labels=["Center","Inner city","Outer city","Suburbs"]
) 

parking_zone = df.groupby("zone")["Parking"].mean()

plt.figure(figsize=(9,6))

parking_zone.plot(
    kind="bar",
    color="#2E86AB"
)

plt.title("Parking Availability by Distance Zone")
plt.ylabel("Share of Apartments with Parking")
plt.xlabel("City Zone")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=df,
    x="zone",
    y="price_per_m2",
    palette="coolwarm"
)

plt.title("Price per m² by Distance Zone")
plt.ylabel("Price per m² (EUR)")
plt.xlabel("City Zone")

plt.show()

In [ ]:
lift_zone = df.groupby("zone")["Lift"].mean()

plt.figure(figsize=(9,6))

lift_zone.plot(
    kind="bar",
    color="#F18F01"
)

plt.title("Lift Availability by Distance Zone")
plt.ylabel("Share of Apartments with Lift")
plt.xlabel("City Zone")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,10))

sns.scatterplot(
    data=df,
    x="lon",
    y="lat",
    hue="price_per_m2",
    palette="coolwarm",
    alpha=0.7
)

plt.title("Belgrade Apartment Prices per m² (Geographical Distribution)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.legend(title="Price per m²", bbox_to_anchor=(1.05,1))

plt.show()

In [ ]:
df.info()

In [ ]:
print(df["Floor"].unique())
# Mapping special floor names to numbers
floor_mapping = {
    "Suteren": -1,
    "Prizemlje": 0,
    "Visoko prizemlje": 0.5
}

# Replace mapped values
df["Floor"] = df["Floor"].replace(floor_mapping)

# Convert the rest to float
df["Floor"] = df["Floor"].astype(float)

In [ ]:
# 1️⃣ Features and target
X = df.drop(columns=["Price_EUR", "Street", "zone","geo_status","price_per_m2"])
y = df["Price_EUR"]

In [ ]:
# 2️⃣ Identify categorical and numerical columns
categorical_cols = ["State"]
numerical_cols = [c for c in X.columns if c not in categorical_cols]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
# 3️⃣ Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough" , # leave numeric columns as-is
     force_int_remainder_cols=False
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Linear regression 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Preprocessing: scale numeric features and one-hot encode categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),  # scale numeric
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)  # one-hot encode
    ]
)


lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train
lr_pipeline.fit(X_train, y_train)

# Predict
y_pred_lr = lr_pipeline.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred_lr)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2 = r2_score(y_test, y_pred_lr)

print(f"Linear Regression -> MAE: {mae:.2f}, RMSE: {rmse:.2f}, R2: {r2:.2f}")

In [ ]:
# Decision Tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"  
)

# Decision Tree pipeline
dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("regressor", DecisionTreeRegressor(max_depth=8, random_state=42))
])

# Train the model
dt_pipeline.fit(X_train, y_train)

# Predict
y_pred_dt = dt_pipeline.predict(X_test)

# Evaluate
mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt = r2_score(y_test, y_pred_dt)

print(f"Decision Tree -> MAE: {mae_dt:.2f}, RMSE: {rmse_dt:.2f}, R2: {r2_dt:.2f}")

In [ ]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

# Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
rf_pipeline.fit(X_train, y_train)

# Predict
y_pred_rf = rf_pipeline.predict(X_test)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest -> MAE: {mae_rf:.2f}, RMSE: {rmse_rf:.2f}, R2: {r2_rf:.2f}")

In [ ]:
# Random forest optimization
from sklearn.model_selection import RandomizedSearchCV

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_tree),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])

# Hyperparameter grid
param_dist = {
    "regressor__n_estimators": [100, 200, 300, 400],
    "regressor__max_depth": [None, 10, 20, 30, 40],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 4],
    "regressor__max_features": [None, "sqrt", "log2"]
}

# Randomized search
random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=30,           # number of random combinations to try
    cv=3,                # 3-fold cross-validation
    scoring="r2",        # optimize for R²
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Run the search
random_search.fit(X_train, y_train)

# Best parameters
print("Best parameters:", random_search.best_params_)

# Evaluate best model
y_pred_rf_best = random_search.best_estimator_.predict(X_test)
mae_best = mean_absolute_error(y_test, y_pred_rf_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_rf_best))
r2_best = r2_score(y_test, y_pred_rf_best)

print(f"Optimized Random Forest -> MAE: {mae_best:.2f}, RMSE: {rmse_best:.2f}, R2: {r2_best:.2f}")

In [ ]:
# XGBoost 
from xgboost import XGBRegressor

# Preprocessor (same as before)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

# Pipeline
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
xgb_pipeline.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_pipeline.predict(X_test)

# Evaluate
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost -> MAE: {mae_xgb:.2f}, RMSE: {rmse_xgb:.2f}, R2: {r2_xgb:.2f}")

In [ ]:
# xgboost optimization

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

# Pipeline
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor_tree),
    ("regressor", XGBRegressor(
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    ))
])

# Enhanced hyperparameter grid
param_dist = {
    "regressor__n_estimators": [200, 300, 400, 500, 600],
    "regressor__max_depth": [4, 5, 6, 7, 8, 10],
    "regressor__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1, 0.2],
    "regressor__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "regressor__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "regressor__gamma": [0, 0.1, 0.2, 0.3],
    "regressor__reg_alpha": [0, 0.01, 0.1, 1],
    "regressor__reg_lambda": [1, 1.5, 2, 3]
}

# Randomized search
random_search_xgb = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_dist,
    n_iter=50,          
    cv=3,
    scoring="r2",
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Run search
random_search_xgb.fit(X_train, y_train)

# Best parameters
print("Best parameters:", random_search_xgb.best_params_)

# Evaluate best model
y_pred_xgb_best = random_search_xgb.best_estimator_.predict(X_test)
mae_best = mean_absolute_error(y_test, y_pred_xgb_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_xgb_best))
r2_best = r2_score(y_test, y_pred_xgb_best)

print(f"Optimized XGBoost -> MAE: {mae_best:.2f}, RMSE: {rmse_best:.2f}, R2: {r2_best:.2f}")

In [ ]:
#Feature importance 
def get_feature_names(preprocessor, numerical_cols, categorical_cols):
    cat_features = preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
    return np.concatenate([cat_features, numerical_cols])

In [ ]:
feature_names = get_feature_names(
    dt_pipeline.named_steps["preprocessor"],
    numerical_cols,
    categorical_cols
)

importances = dt_pipeline.named_steps["regressor"].feature_importances_

feat_imp_dt = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp_dt.head(10))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=feat_imp_dt.head(15),
    x="importance",
    y="feature"
)
plt.title("Top 15 Features (Decision Tree)")
plt.show()

In [ ]:
#Random forest
# Get feature names
best_rf = random_search.best_estimator_

feature_names = get_feature_names(
    best_rf.named_steps["preprocessor"],
    numerical_cols,
    categorical_cols
)

importances = best_rf.named_steps["regressor"].feature_importances_

# Create dataframe
feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp.head(10))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=feat_imp.head(15),
    x="importance",
    y="feature"
)
plt.title("Top 15 Features (Random Forest)")
plt.show()

In [ ]:
#Xgboost 
# Get feature names
best_xgb = random_search_xgb.best_estimator_

feature_names = get_feature_names(
    best_xgb.named_steps["preprocessor"],
    numerical_cols,
    categorical_cols
)

# Get importances
importances = best_xgb.named_steps["regressor"].feature_importances_

# Create dataframe
feat_imp_xgb = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp_xgb.head(10))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=feat_imp_xgb.head(15),
    x="importance",
    y="feature"
)
plt.title("Top 15 Features (XGBoost)")
plt.show()

In [ ]:
# Linear regression: 
feature_names = lr_pipeline.named_steps["preprocessor"].get_feature_names_out()

coefficients = lr_pipeline.named_steps["regressor"].coef_

feat_imp_lr = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

# Sort by absolute importance
feat_imp_lr["abs_coef"] = feat_imp_lr["coefficient"].abs()
feat_imp_lr = feat_imp_lr.sort_values(by="abs_coef", ascending=False)

print(feat_imp_lr.head(10))

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(
    data=feat_imp_lr.head(15),
    x="abs_coef",
    y="feature"
)
plt.title("Top 15 Feature Importance (Linear Regression)")
plt.xlabel("Importance (Absolute Coefficient)")
plt.show()